# Red Siamesa (Contrastive Loss) — Clasificación de hojas

Este notebook implementa el escenario experimental basado en una **red siamesa entrenada
con contrastive loss**, sobre el mismo dataset de hojas (`train / valid / test`) usado en
`clasificacion_hojas_resnet_vs_vit.ipynb`.

**Protocolo (según la consigna del curso):**
1. Entrenar una red siamesa con **contrastive loss** para separar las muestras por clase
   (aprender un espacio de embeddings donde imágenes de la misma clase queden cerca y de
   clases distintas queden lejos).
2. **Tarea A:** congelar el backbone entrenado, añadir una capa **FC** y entrenar esa capa
   para clasificar las clases del dataset.
3. **Tarea B:** usar el mismo backbone congelado para extraer **vectores característicos**
   (embeddings) de cada imagen, y entrenar un modelo de **ML (XGBoost)** sobre esos vectores
   para clasificar las categorías.

Al final se comparan ambas tareas (FC-head vs. XGBoost) usando las mismas métricas del
notebook de ResNet/ViT (accuracy, precision/recall/F1 macro, F1 ponderado, matriz de
confusión), para poder integrarlas en el informe final.


## 0. Dependencias

Si te falta algún paquete, descomenta y ejecuta la siguiente celda.

In [ ]:
!pip install torch torchvision scikit-learn matplotlib seaborn pandas tqdm xgboost


## 1. Imports y configuración

In [ ]:
import os
import time
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from sklearn.decomposition import PCA

import xgboost as xgb

# tqdm.auto detecta automáticamente el entorno (notebook o consola)
# y no requiere ipywidgets funcionando para caer a la versión de texto
from tqdm.auto import tqdm


In [ ]:
# ----------------------------- CONFIGURACIÓN -----------------------------
DATA_DIR = Path("Dataset_split_Aug")          # debe contener train/ valid/ test/

IMG_SIZE     = 224
BATCH_SIZE   = 32                   # baja a 16 si te quedas sin memoria de GPU
NUM_WORKERS  = 0 if os.name == "nt" else 4   # 0 en Windows (evita bloqueos de multiprocessing en Jupyter)
SEED = 42

# --- Red siamesa (contrastive loss) ---
EMBED_DIM        = 128              # dimensión del espacio de embeddings
MARGIN           = 1.0              # margen del contrastive loss
EPOCHS_SIAMESE   = 10
LR_SIAMESE       = 3e-4
WEIGHT_DECAY     = 1e-4
PAIRS_PER_EPOCH_TRAIN = None        # se define tras cargar los datasets (2x tamaño de train)
PAIRS_VALID_FIXED     = 2000        # pares fijos de validación (reproducibles)

# --- Cabeza FC sobre backbone congelado (Tarea A) ---
EPOCHS_CLF = 10
LR_CLF     = 1e-3
LABEL_SMOOTHING = 0.1

OUTPUT_DIR = Path("checkpoints")
OUTPUT_DIR.mkdir(exist_ok=True)

# Reproducibilidad
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# Con tamaño de imagen fijo, benchmark=True deja que cuDNN elija los kernels más rápidos
torch.backends.cudnn.benchmark = torch.cuda.is_available()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = (DEVICE == "cuda")
print("Dispositivo:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Transforms y Datasets

Usamos los mismos `train / valid / test` que el notebook de ResNet/ViT. Además creamos
una versión **sin aumentos** del set de entrenamiento (`train_ds_eval`), necesaria para
extraer embeddings estables (sin aleatoriedad) en la Tarea B.

In [ ]:
# Normalización estándar de ImageNet (los pesos preentrenados la esperan)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds      = datasets.ImageFolder(DATA_DIR / "train", transform=train_tf)   # con aumentos (para entrenar)
train_ds_eval = datasets.ImageFolder(DATA_DIR / "train", transform=eval_tf)    # sin aumentos (para extraer embeddings)
valid_ds      = datasets.ImageFolder(DATA_DIR / "valid", transform=eval_tf)
test_ds       = datasets.ImageFolder(DATA_DIR / "test",  transform=eval_tf)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)

pin = (DEVICE == "cuda")
# kwargs comunes: persistent_workers evita re-crear procesos en cada época (solo aplica si hay workers)
loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=pin,
                 persistent_workers=(NUM_WORKERS > 0))
train_loader      = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               **loader_kw)
train_eval_loader = DataLoader(train_ds_eval, batch_size=BATCH_SIZE, shuffle=False,
                               **loader_kw)
valid_loader      = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                               **loader_kw)
test_loader       = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               **loader_kw)

print(f"Clases ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Imágenes -> train: {len(train_ds)} | valid: {len(valid_ds)} | test: {len(test_ds)}")


## 3. Dataset de pares (para contrastive loss)

Cada muestra es un **par de imágenes** (img1, img2) con una etiqueta binaria:
`label = 1` si ambas pertenecen a la **misma clase** (par similar) y `label = 0` si son de
**clases distintas** (par disimilar), con ~50/50 de probabilidad. Para `valid` generamos un
conjunto de pares **fijo** (semilla fija) para que la validación sea reproducible entre épocas.

In [ ]:
class PairDataset(Dataset):
    """Genera pares (img1, img2, label) a partir de un ImageFolder.
    label = 1.0 -> misma clase (par similar)
    label = 0.0 -> clases distintas (par disimilar)
    """
    def __init__(self, base_dataset, n_pairs, fixed=False, seed=None):
        self.base = base_dataset
        self.targets = np.array(base_dataset.targets)
        self.classes = np.unique(self.targets)
        self.class_to_indices = {c: np.where(self.targets == c)[0] for c in self.classes}
        self.n_pairs = n_pairs
        self.fixed = fixed
        if fixed:
            rng = np.random.RandomState(seed)
            self.pairs = [self._sample_pair(rng) for _ in range(n_pairs)]

    def _sample_pair(self, rng):
        idx1 = int(rng.randint(len(self.base)))
        y1 = self.targets[idx1]
        same = rng.rand() < 0.5
        if same:
            # positive: misma clase pero índice distinto al de img1 (si la clase tiene >1 imagen),
            # para no entrenar con pares triviales de la misma imagen consigo misma
            same_idx = self.class_to_indices[y1]
            if len(same_idx) > 1:
                idx2 = idx1
                while idx2 == idx1:
                    idx2 = int(rng.choice(same_idx))
            else:
                idx2 = idx1
        else:
            other_classes = self.classes[self.classes != y1]
            y2 = rng.choice(other_classes)
            idx2 = int(rng.choice(self.class_to_indices[y2]))
        return idx1, idx2, (1.0 if same else 0.0)

    def __len__(self):
        return self.n_pairs

    def __getitem__(self, i):
        if self.fixed:
            idx1, idx2, label = self.pairs[i]
        else:
            idx1, idx2, label = self._sample_pair(np.random)
        img1, _ = self.base[idx1]
        img2, _ = self.base[idx2]
        return img1, img2, torch.tensor(label, dtype=torch.float32)


PAIRS_PER_EPOCH_TRAIN = len(train_ds) * 2

train_pair_ds = PairDataset(train_ds, n_pairs=PAIRS_PER_EPOCH_TRAIN, fixed=False)
valid_pair_ds = PairDataset(valid_ds, n_pairs=PAIRS_VALID_FIXED, fixed=True, seed=SEED)

train_pair_loader = DataLoader(train_pair_ds, batch_size=BATCH_SIZE, shuffle=True,
                               **loader_kw)
valid_pair_loader = DataLoader(valid_pair_ds, batch_size=BATCH_SIZE, shuffle=False,
                               **loader_kw)

print(f"Pares de entrenamiento por época: {len(train_pair_ds)} | Pares de validación (fijos): {len(valid_pair_ds)}")


## 4. Arquitectura de la red siamesa

Backbone **ResNet-50** preentrenado en ImageNet (compartido entre ambas ramas: los mismos
pesos procesan `img1` e `img2`), seguido de una pequeña cabeza de proyección a un espacio
de embeddings de dimensión `EMBED_DIM`, normalizado con L2 (estándar en contrastive/triplet
learning).

In [ ]:
class SiameseNetwork(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        backbone = models.resnet50(weights=weights)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()      # quitamos la cabeza de clasificación original
        self.backbone = backbone
        self.embedding = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, embed_dim),
        )

    def forward_once(self, x):
        feat = self.backbone(x)
        emb = self.embedding(feat)
        emb = F.normalize(emb, p=2, dim=1)   # embeddings en la hiperesfera unitaria
        return emb

    def forward(self, x1, x2):
        return self.forward_once(x1), self.forward_once(x2)


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 5. Contrastive Loss

Función de pérdida de Hadsell et al. (2006):

$$L = \tfrac12\, y\, d^2 \;+\; \tfrac12\,(1-y)\,\max(0,\, m - d)^2$$

donde $d$ es la distancia euclidiana entre los embeddings del par, $y=1$ si el par es de la
misma clase (se penaliza que estén lejos) e $y=0$ si son de clases distintas (se penaliza que
estén más cerca que el margen $m$).

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        dist = F.pairwise_distance(emb1, emb2)
        loss_similar    = label * dist.pow(2)
        loss_dissimilar = (1 - label) * torch.clamp(self.margin - dist, min=0).pow(2)
        loss = 0.5 * (loss_similar + loss_dissimilar)
        return loss.mean(), dist


## 6. Entrenamiento de la red siamesa

In [ ]:
def train_siamese(model, epochs=EPOCHS_SIAMESE, lr=LR_SIAMESE, margin=MARGIN, name="siamese_contrastive"):
    model = model.to(DEVICE)
    criterion = ContrastiveLoss(margin=margin)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

    history = {"train_loss": [], "val_loss": [], "val_pair_acc": []}
    best_val_loss, best_state = float("inf"), None
    ckpt_path = OUTPUT_DIR / f"best_{name}.pt"

    print(f"\n===== Entrenando {name} | params entrenables: {count_trainable(model):,} =====")
    for epoch in range(1, epochs + 1):
        # ---------------- Train ----------------
        model.train()
        run_loss, seen = 0.0, 0
        pbar = tqdm(train_pair_loader, desc=f"[{name}] Época {epoch}/{epochs}", leave=False)
        for x1, x2, y in pbar:
            x1, x2, y = x1.to(DEVICE, non_blocking=True), x2.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=USE_AMP):
                emb1, emb2 = model(x1, x2)
                loss, _ = criterion(emb1, emb2, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # estabiliza el entrenamiento con AMP
            scaler.step(optimizer)
            scaler.update()

            run_loss += loss.item() * x1.size(0)
            seen += x1.size(0)
            pbar.set_postfix(loss=f"{run_loss/seen:.4f}")

        scheduler.step()
        train_loss = run_loss / seen

        # ---------------- Validación ----------------
        model.eval()
        val_run_loss, val_seen, val_correct = 0.0, 0, 0
        with torch.no_grad():
            for x1, x2, y in valid_pair_loader:
                x1, x2, y = x1.to(DEVICE), x2.to(DEVICE), y.to(DEVICE)
                emb1, emb2 = model(x1, x2)
                loss, dist = criterion(emb1, emb2, y)
                val_run_loss += loss.item() * x1.size(0)
                # accuracy de pares: predecimos "misma clase" si dist < margin/2
                pred_same = (dist < margin / 2).float()
                val_correct += (pred_same == y).sum().item()
                val_seen += x1.size(0)
        val_loss = val_run_loss / val_seen
        val_pair_acc = val_correct / val_seen

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_pair_acc"].append(val_pair_acc)
        print(f"Época {epoch:2d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} "
              f"| val pair-acc {val_pair_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, ckpt_path)

    print(f"Mejor val_loss ({name}): {best_val_loss:.4f}  ->  {ckpt_path}")
    model.load_state_dict(best_state)
    return model, history


In [ ]:
set_seed(SEED)
siamese = SiameseNetwork(embed_dim=EMBED_DIM)
t0 = time.time()
siamese, hist_siamese = train_siamese(siamese, epochs=EPOCHS_SIAMESE, lr=LR_SIAMESE, margin=MARGIN)
time_siamese = time.time() - t0
print(f"Tiempo de entrenamiento red siamesa (contrastive): {time_siamese/60:.1f} min")


### 6.1 Curva de pérdida

In [ ]:
plt.figure(figsize=(6, 4))
epochs_range = range(1, len(hist_siamese["train_loss"]) + 1)
plt.plot(epochs_range, hist_siamese["train_loss"], "-o", label="train")
plt.plot(epochs_range, hist_siamese["val_loss"], "-o", label="valid")
plt.title("Contrastive loss — red siamesa"); plt.xlabel("Época"); plt.ylabel("Loss")
plt.legend(); plt.tight_layout(); plt.show()


### 6.2 Visualización de embeddings y separación de clases

Proyectamos los embeddings del set de validación a 2D con PCA y medimos, sobre los pares
fijos de validación, la distancia promedio entre pares de la **misma clase** vs. pares de
**clases distintas**: si el entrenamiento funcionó, la primera debe ser claramente menor que
la segunda.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader):
    model.eval()
    feats, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        emb = model.forward_once(x)
        feats.append(emb.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)


# --- Proyección PCA de embeddings de validación ---
X_valid_emb, y_valid_emb = extract_embeddings(siamese, valid_loader)
pca = PCA(n_components=2, random_state=SEED)
X_valid_2d = pca.fit_transform(X_valid_emb)

plt.figure(figsize=(7, 6))
scatter = plt.scatter(X_valid_2d[:, 0], X_valid_2d[:, 1], c=y_valid_emb, cmap="tab20", s=10)
plt.title("Embeddings de validación (PCA 2D) — red siamesa contrastive")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.colorbar(scatter, label="Clase (índice)")
plt.tight_layout(); plt.show()

# --- Distancia intra-clase vs. inter-clase sobre los pares fijos de validación ---
siamese.eval()
dists_same, dists_diff = [], []
with torch.no_grad():
    for x1, x2, y in valid_pair_loader:
        x1, x2, y = x1.to(DEVICE), x2.to(DEVICE), y.to(DEVICE)
        emb1, emb2 = siamese(x1, x2)
        d = F.pairwise_distance(emb1, emb2).cpu().numpy()
        y = y.cpu().numpy()
        dists_same.extend(d[y == 1]); dists_diff.extend(d[y == 0])

print(f"Distancia promedio — misma clase:   {np.mean(dists_same):.4f}")
print(f"Distancia promedio — clases distintas: {np.mean(dists_diff):.4f}")


## 7. Tarea A — Clasificador FC sobre backbone congelado

Se congela el backbone de la red siamesa (pesos fijos, `eval()` permanente para las
capas `BatchNorm`) y se añade una única capa lineal (`FC`) que mapea el embedding a las
`NUM_CLASSES` categorías. Solo esta capa se entrena.

In [ ]:
class FrozenSiameseClassifier(nn.Module):
    def __init__(self, siamese_model, num_classes, embed_dim):
        super().__init__()
        self.siamese = siamese_model
        for p in self.siamese.parameters():
            p.requires_grad = False
        self.head = nn.Linear(embed_dim, num_classes)

    def train(self, mode=True):
        # El backbone siamés se mantiene siempre en eval() (BatchNorm congelado),
        # sin importar si el módulo completo está en modo train o eval.
        super().train(mode)
        self.siamese.eval()
        return self

    def forward(self, x):
        with torch.no_grad():
            emb = self.siamese.forward_once(x)
        return self.head(emb)


In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion=None):
    """Devuelve loss, accuracy, f1_macro y las predicciones/etiquetas completas."""
    model.eval()
    all_preds, all_labels = [], []
    total_loss, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(x)
        if criterion is not None:
            total_loss += criterion(logits, y).item() * x.size(0)
        preds = logits.argmax(1)
        all_preds.append(preds.cpu()); all_labels.append(y.cpu())
        n += x.size(0)
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()
    loss = total_loss / n if criterion is not None else None
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return loss, acc, f1m, y_true, y_pred


def train_model(model, name, epochs, lr):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

    history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "val_f1"]}
    best_f1, best_state = -1.0, None
    ckpt_path = OUTPUT_DIR / f"best_{name}.pt"

    print(f"\n===== Entrenando {name} | params entrenables: {count_trainable(model):,} =====")
    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, run_correct, seen = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"[{name}] Época {epoch}/{epochs}", leave=False)
        for x, y in pbar:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            run_loss += loss.item() * x.size(0)
            run_correct += (logits.argmax(1) == y).sum().item()
            seen += x.size(0)
            pbar.set_postfix(loss=f"{run_loss/seen:.3f}", acc=f"{run_correct/seen:.3f}")

        scheduler.step()
        train_loss = run_loss / seen
        train_acc = run_correct / seen

        val_loss, val_acc, val_f1, _, _ = evaluate(model, valid_loader, criterion)
        for k, v in zip(history, [train_loss, train_acc, val_loss, val_acc, val_f1]):
            history[k].append(v)
        print(f"Época {epoch:2d} | train_loss {train_loss:.3f} acc {train_acc:.3f} "
              f"| val_loss {val_loss:.3f} acc {val_acc:.3f} f1 {val_f1:.3f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, ckpt_path)

    print(f"Mejor F1-macro de validación ({name}): {best_f1:.4f}  ->  {ckpt_path}")
    model.load_state_dict(best_state)
    return model, history


In [ ]:
set_seed(SEED)
clf_fc = FrozenSiameseClassifier(siamese, NUM_CLASSES, EMBED_DIM)
t0 = time.time()
clf_fc, hist_fc = train_model(clf_fc, "siamese_contrastive_fc", epochs=EPOCHS_CLF, lr=LR_CLF)
time_fc = time.time() - t0
print(f"Tiempo de entrenamiento cabeza FC: {time_fc/60:.1f} min")


### 7.1 Evaluación en test (Tarea A)

In [ ]:
def full_test_report(model, name, loader=test_loader):
    _, acc, f1m, y_true, y_pred = evaluate(model, loader)
    prec_m = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec_m  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_w   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n########## {name} — resultados en TEST ##########")
    print(f"Accuracy         : {acc:.4f}")
    print(f"Precision (macro): {prec_m:.4f}")
    print(f"Recall    (macro): {rec_m:.4f}")
    print(f"F1        (macro): {f1m:.4f}")
    print(f"F1     (weighted): {f1_w:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    metrics = {"model": name, "accuracy": acc, "precision_macro": prec_m,
               "recall_macro": rec_m, "f1_macro": f1m, "f1_weighted": f1_w}
    return metrics, y_true, y_pred


metrics_fc, yt_fc, yp_fc = full_test_report(clf_fc, "Siamese-Contrastive + FC")


## 8. Tarea B — Extracción de embeddings + XGBoost

Usamos el **mismo backbone siamés congelado** (sin la capa FC) para extraer un vector
característico por imagen, y entrenamos un `XGBClassifier` sobre esos vectores.

In [ ]:
X_train_emb, y_train_emb = extract_embeddings(siamese, train_eval_loader)
X_valid_emb, y_valid_emb = extract_embeddings(siamese, valid_loader)
X_test_emb,  y_test_emb  = extract_embeddings(siamese, test_loader)

print("Embeddings ->", "train:", X_train_emb.shape, "valid:", X_valid_emb.shape, "test:", X_test_emb.shape)


In [ ]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=NUM_CLASSES,
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    early_stopping_rounds=20,
)

t0 = time.time()
xgb_clf.fit(
    X_train_emb, y_train_emb,
    eval_set=[(X_valid_emb, y_valid_emb)],
    verbose=False,
)
time_xgb = time.time() - t0
print(f"Tiempo de entrenamiento XGBoost: {time_xgb:.1f} s | mejor iteración: {xgb_clf.best_iteration}")

xgb_clf.save_model(OUTPUT_DIR / "xgb_siamese_contrastive.json")


### 8.1 Evaluación en test (Tarea B)

In [ ]:
def evaluate_predictions(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec_m  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1m    = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_w   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n########## {name} — resultados en TEST ##########")
    print(f"Accuracy         : {acc:.4f}")
    print(f"Precision (macro): {prec_m:.4f}")
    print(f"Recall    (macro): {rec_m:.4f}")
    print(f"F1        (macro): {f1m:.4f}")
    print(f"F1     (weighted): {f1_w:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    metrics = {"model": name, "accuracy": acc, "precision_macro": prec_m,
               "recall_macro": rec_m, "f1_macro": f1m, "f1_weighted": f1_w}
    return metrics


y_pred_xgb = xgb_clf.predict(X_test_emb)
metrics_xgb = evaluate_predictions(y_test_emb, y_pred_xgb, "Siamese-Contrastive + XGBoost")


## 9. Comparación de resultados

In [ ]:
summary = pd.DataFrame([metrics_fc, metrics_xgb]).set_index("model")
summary["train_time_min"] = [time_fc / 60, time_xgb / 60]
summary["trainable_params_M"] = [count_trainable(clf_fc) / 1e6, np.nan]

display(summary.round(4))

summary.round(4).to_csv(OUTPUT_DIR / "comparacion_metricas_siamese_contrastive.csv")
print("Métricas guardadas en", OUTPUT_DIR / "comparacion_metricas_siamese_contrastive.csv")

# Si ya corriste el notebook de ResNet/ViT, esto añade esos resultados como contexto adicional
baseline_path = OUTPUT_DIR / "comparacion_metricas.csv"
if baseline_path.exists():
    baseline = pd.read_csv(baseline_path, index_col=0)
    combined = pd.concat([baseline[summary.columns.intersection(baseline.columns)], summary])
    print("\nComparación combinada con CNN/ViT (checkpoints/comparacion_metricas.csv):")
    display(combined.round(4))


In [ ]:
metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"]
ax = summary[metric_cols].T.plot(kind="bar", figsize=(11, 4.5), color=["#2ca02c", "#ff7f0e"])
ax.set_title("Siamese-Contrastive: FC-head vs. XGBoost — métricas en test")
ax.set_ylabel("Puntaje"); ax.set_ylim(0, 1)
ax.legend(title="Modelo"); plt.xticks(rotation=20, ha="right")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8, padding=2)
plt.tight_layout(); plt.show()


### 9.1 Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(2 * max(6, NUM_CLASSES * 0.7), max(5, NUM_CLASSES * 0.6)))
for ax, (yt, yp, title) in zip(
        axes,
        [(yt_fc, yp_fc, "Siamese-Contrastive + FC"), (y_test_emb, y_pred_xgb, "Siamese-Contrastive + XGBoost")]):
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"Matriz de confusión — {title}")
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
    ax.tick_params(axis="x", rotation=45); ax.tick_params(axis="y", rotation=0)
plt.tight_layout(); plt.show()


## 10. Conclusiones

Rellena esta sección con tus observaciones a partir de los resultados anteriores. Algunas
preguntas guía para tu informe:

- **¿La contrastive loss logró separar las clases?** Revisa la sección 6.2: ¿la distancia
  intra-clase fue claramente menor que la inter-clase? ¿Se ven agrupamientos por clase en
  la proyección PCA?
- **¿FC-head o XGBoost obtuvo mejor F1-macro / accuracy en test?** ¿Tiene sentido dado que
  ambos usan exactamente los mismos embeddings, solo cambia el clasificador final?
- **¿Cómo se compara este escenario (siamese-contrastive) con el CNN (ResNet-50) y el ViT-B/16
  entrenados end-to-end** en `clasificacion_hojas_resnet_vs_vit.ipynb`? ¿Vale la pena el costo
  extra de entrenar la red siamesa, o el fine-tuning directo ya es suficiente?
- **¿En qué clases falla más cada clasificador** (revisa las matrices de confusión y compáralas
  con las del CNN/ViT)? ¿Son las mismas clases difíciles en todos los escenarios?
- Este notebook cubre la variante **contrastive**; el siguiente paso es repetir el mismo
  protocolo con **triplet loss** (`clasificacion_hojas_siamese_triplet.ipynb`) para completar
  la comparación de los 6 escenarios experimentales pedidos en la consigna.
